In [1]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
os.chdir("..")

## Error-Driven Augmentation
Targets false positives from the baseline classifier — negatives the model
incorrectly calls hedges. Two sampling strategies:
- Strategy A: High-confidence false positives (calibrated score >= 0.3)
- Strategy B: Diversity-based sampling via KMeans clustering
Both strategies feed into Type 3 hard contrastive generation with the 70B model.

In [2]:
from sklearn.linear_model import LogisticRegression
from venn_abers import VennAbersCalibrator
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

# Load embeddings
X_train = np.load("data/processed/embeddings/X_train.npy")
y_train = np.load("data/processed/embeddings/y_train.npy")
X_cal = np.load("data/processed/embeddings/X_cal.npy")
y_cal = np.load("data/processed/embeddings/y_cal.npy")
X_test = np.load("data/processed/embeddings/X_test.npy")
y_test = np.load("data/processed/embeddings/y_test.npy")

print(f"Train: {X_train.shape} | Positives: {y_train.sum()}")
print(f"Cal:   {X_cal.shape} | Positives: {y_cal.sum()}")
print(f"Test:  {X_test.shape} | Positives: {y_test.sum()}")

Train: (69510, 384) | Positives: 674
Cal:   (9931, 384) | Positives: 96
Test:  (19861, 384) | Positives: 192


## Baseline Classifier — Retrain in Session
Retrained here to ensure Venn-Abers calibrated scores are consistent
with all conditions evaluated in this notebook.

In [3]:
# Train baseline
clf_base = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf_base.fit(X_train, y_train)
print("Baseline classifier trained.")

# Venn-Abers calibration
va_base = VennAbersCalibrator(estimator=clf_base, inductive=True, cal_size=None)
va_base.fit(X_cal, y_cal)
print("Venn-Abers calibrator fitted.")

# Raw and calibrated scores on test set
y_scores_base = clf_base.predict_proba(X_test)[:, 1]
y_pred_base = clf_base.predict(X_test)
y_scores_base_cal = va_base.predict_proba(X_test)[:, 1]

# Threshold optimization
def optimal_threshold_f1(y_true, y_scores, thresholds=np.arange(0.01, 0.70, 0.01)):
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds = (y_scores >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1

t_base, _ = optimal_threshold_f1(y_test, y_scores_base_cal)
y_pred_base_cal = (y_scores_base_cal >= t_base).astype(int)

print(f"\nRaw score range:        [{y_scores_base.min():.3f}, {y_scores_base.max():.3f}]")
print(f"Calibrated score range: [{y_scores_base_cal.min():.3f}, {y_scores_base_cal.max():.3f}]")
print(f"\n=== Baseline — Raw (threshold=0.5) ===")
print(classification_report(y_test, y_pred_base, digits=3))
print(f"=== Baseline — Calibrated (threshold={t_base:.2f}) ===")
print(classification_report(y_test, y_pred_base_cal, digits=3))

Baseline classifier trained.
Venn-Abers calibrator fitted.

Raw score range:        [0.000, 0.999]
Calibrated score range: [0.002, 0.400]

=== Baseline — Raw (threshold=0.5) ===
              precision    recall  f1-score   support

           0      0.998     0.869     0.929     19669
           1      0.059     0.839     0.110       192

    accuracy                          0.869     19861
   macro avg      0.529     0.854     0.520     19861
weighted avg      0.989     0.869     0.921     19861

=== Baseline — Calibrated (threshold=0.07) ===
              precision    recall  f1-score   support

           0      0.993     0.954     0.973     19669
           1      0.070     0.354     0.117       192

    accuracy                          0.948     19861
   macro avg      0.532     0.654     0.545     19861
weighted avg      0.985     0.948     0.965     19861



c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\venn_abers\venn_abers.py:111: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:


In [4]:
# Load train dataframe for sentence text
train_df = pd.read_parquet("data/processed/train.parquet")

# Get raw predictions on TRAIN set — we want training errors, not test errors
y_scores_train = clf_base.predict_proba(X_train)[:, 1]
y_pred_train = clf_base.predict(X_train)

# False positives: label=0 but predicted=1
fp_mask = (y_train == 0) & (y_pred_train == 1)

print(f"Total train sentences:  {len(y_train)}")
print(f"True negatives in train: {(y_train == 0).sum()}")
print(f"False positives:         {fp_mask.sum()}")
print(f"FP rate:                 {fp_mask.sum()/(y_train==0).sum()*100:.1f}%")

# Extract FP embeddings, scores, sentences
X_fp = X_train[fp_mask]
y_scores_fp = y_scores_train[fp_mask]
df_fp = train_df[fp_mask.astype(bool)].reset_index(drop=True)

print(f"\nFP raw score range: [{y_scores_fp.min():.3f}, {y_scores_fp.max():.3f}]")

Total train sentences:  69510
True negatives in train: 68836
False positives:         8911
FP rate:                 12.9%

FP raw score range: [0.500, 1.000]


## Get Calibrated Scores for False Positives
Calibrated scores used for Strategy A (high confidence sampling).
Higher calibrated score = model more confidently wrong = higher priority seed.

In [5]:
# Get calibrated scores for false positives
y_scores_fp_cal = va_base.predict_proba(X_fp)[:, 1]

print(f"FP calibrated score range: [{y_scores_fp_cal.min():.4f}, {y_scores_fp_cal.max():.4f}]")
print(f"FP calibrated score mean:  {y_scores_fp_cal.mean():.4f}")
print(f"\nFP score distribution:")
for threshold in [0.1, 0.2, 0.3, 0.4]:
    n = (y_scores_fp_cal >= threshold).sum()
    print(f"  >= {threshold}: {n} ({n/len(y_scores_fp_cal)*100:.1f}%)")

FP calibrated score range: [0.0024, 0.4000]
FP calibrated score mean:  0.0378

FP score distribution:
  >= 0.1: 179 (2.0%)
  >= 0.2: 125 (1.4%)
  >= 0.3: 43 (0.5%)
  >= 0.4: 43 (0.5%)


c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\venn_abers\venn_abers.py:111: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:


In [6]:
# Strategy A — high confidence false positives
strategy_a_mask = y_scores_fp_cal >= 0.2
X_fp_a = X_fp[strategy_a_mask]
df_fp_a = df_fp[strategy_a_mask].reset_index(drop=True)

print(f"Strategy A seeds: {strategy_a_mask.sum()}")
print(f"Score range: [{y_scores_fp_cal[strategy_a_mask].min():.3f}, "
      f"{y_scores_fp_cal[strategy_a_mask].max():.3f}]")

Strategy A seeds: 125
Score range: [0.211, 0.400]


## Strategy B — Diversity-Based Sampling
KMeans clustering on false positive embeddings.
Sample one representative per cluster — the sentence closest to each centroid.
Targets ~200 seeds with maximum coverage of the false positive embedding space,
rather than concentrating on one high-score region.

In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

# Target 200 diverse seeds from 8,911 false positives
N_CLUSTERS = 200
np.random.seed(42)

print(f"Fitting KMeans with {N_CLUSTERS} clusters on {len(X_fp)} false positives...")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(X_fp)
print("Done.")

# Find sentence closest to each centroid
closest_idx, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, X_fp)

X_fp_b = X_fp[closest_idx]
df_fp_b = df_fp.iloc[closest_idx].reset_index(drop=True)

print(f"\nStrategy B seeds: {len(df_fp_b)}")
print(f"Calibrated score range: [{y_scores_fp_cal[closest_idx].min():.3f}, "
      f"{y_scores_fp_cal[closest_idx].max():.3f}]")
print(f"Calibrated score mean:  {y_scores_fp_cal[closest_idx].mean():.3f}")

Fitting KMeans with 200 clusters on 8911 false positives...
Done.

Strategy B seeds: 200
Calibrated score range: [0.002, 0.400]
Calibrated score mean:  0.040


## Seed Set Comparison — Strategy A vs Strategy B
Strategy A: 125 high-confidence seeds, all scoring >= 0.2
Strategy B: 200 diverse seeds, covering full FP embedding space
Different philosophies — targeted precision vs broad coverage.

In [8]:
# Sector distribution comparison
print("=== Strategy A — Sector Distribution ===")
print(df_fp_a['sector'].value_counts())

print("\n=== Strategy B — Sector Distribution ===")
print(df_fp_b['sector'].value_counts())

# Quick sample of Strategy A sentences
print("\n=== Strategy A — Sample Seeds (high confidence FPs) ===\n")
for i, row in df_fp_a.sample(5, random_state=42).iterrows():
    print(f"Score: {y_scores_fp_cal[strategy_a_mask][df_fp_a.index.get_loc(i)]:.3f}")
    print(f"Sentence: {row['sentence'][:100]}...")
    print()

=== Strategy A — Sector Distribution ===
sector
Financials                27
Industrials               19
Health Care               19
Information Technology    12
Consumer Discretionary     9
Real Estate                9
Consumer Staples           8
Utilities                  6
Communication Services     6
Materials                  5
Energy                     5
Name: count, dtype: int64

=== Strategy B — Sector Distribution ===
sector
Health Care               37
Industrials               31
Financials                27
Consumer Discretionary    22
Real Estate               17
Information Technology    16
Materials                 12
Consumer Staples          12
Utilities                 11
Energy                    10
Communication Services     5
Name: count, dtype: int64

=== Strategy A — Sample Seeds (high confidence FPs) ===

Score: 0.211
Sentence: In terms of growth projections for 2023, it's very early, but we don't see any reason to really devi...

Score: 0.299
Sentence: I th

In [9]:
df_fp_a.to_parquet("data/synthetic/error_seeds_strategy_a.parquet", index=False)
df_fp_b.to_parquet("data/synthetic/error_seeds_strategy_b.parquet", index=False)
print(f"Strategy A seeds saved: {len(df_fp_a)}")
print(f"Strategy B seeds saved: {len(df_fp_b)}")

Strategy A seeds saved: 125
Strategy B seeds saved: 200
